# Portfolio Risk Dashboard

Cell 1 runs the pipeline and prints data validation results (ticker consistency, missing values, exposure reconciliation), KPIs, exposure breakdowns, and stress scenario results. Cell 2 renders the interactive dashboard below: adjust the date range or exposure breakdown and the chart updates immediately. The stress sensitivity surface is point-in-time (based on today's security data) and does not change with either control.

In [1]:
from main import main
from src.dashboard import build_dashboard_page
from src.stress import compute_stress_surface
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

results = main()



=== Data Validation ===

Ticker consistency across SecurityMaster, PortfolioWeights, and TotalReturnIndex_Local: confirmed.

No duplicate tickers in SecurityMaster or PortfolioWeights: confirmed.

Missing values: 3 out of 156,600 possible security-day observations.
      Date  Ticker
2024-06-04 RISK023
2024-11-12 RISK145
2025-07-01 RISK269

Loan pricing: prices are flat 80% of days (all loans update same day each week).
Naive daily vol 1.72% vs weekly-resampled 1.74% (median per-loan): roughly equal, no adjustment needed.

Exposure reconciliation vs. embedded answer key:
       Computed  Stated
Net      1.0000  1.0000
Gross    1.1000  1.1000
Short   -0.0500 -0.0500

EXCLUDED: 1 day(s) below 95% coverage, return set to NaN:
Date
2024-01-02    0.0
dtype: float64
Portfolio returns: 521 valid observations out of 522 total days.

=== KPI Summary ===

Cumulative Return:       -1.55%
Annualized Volatility:   8.89%
95% VaR (1-day):         0.96%
99% VaR (1-day):         1.25%
95% Expected Sho

In [2]:

weights = results['data']['PortfolioWeights']
security_master = results['data']['SecurityMaster']
portfolio_returns = results['portfolio_returns']

# Computed once: the stress surface doesn't depend on the date range
# or exposure dimension, so it's computed here 
equity_shocks = np.linspace(0, -0.25, 26)
rate_shocks_bps = np.linspace(0, 200, 21)
stress_grid = compute_stress_surface(security_master, weights, equity_shocks, rate_shocks_bps)

min_date = portfolio_returns.index.min().date()
max_date = portfolio_returns.index.max().date()

start_picker = widgets.DatePicker(description='Start:', value=min_date)
end_picker = widgets.DatePicker(description='End:', value=max_date)
dimension_dropdown = widgets.Dropdown(
    options=['Asset Type', 'Sector', 'Country', 'Rating'],
    value='Asset Type',
    description='Exposure:'
)
run_button = widgets.Button(description='Recalculate', button_style='primary')
dashboard_output = widgets.Output()

def render_dashboard(b=None):
    with dashboard_output:
        clear_output(wait=True)
        start = pd.Timestamp(start_picker.value)
        end = pd.Timestamp(end_picker.value)
        if start >= end:
            print("Start date must be before end date.")
            return
        filtered_returns = portfolio_returns.loc[start:end]
        if filtered_returns.notna().sum() < 20:
            print(f"Only {filtered_returns.notna().sum()} valid observation(s) in this range, "
                  f"too few for stable VaR/ES/volatility estimates. Choose a wider range.")
            return
        fig = build_dashboard_page(filtered_returns, weights, equity_shocks, rate_shocks_bps, stress_grid,
                                    exposure_dimension=dimension_dropdown.value)
        plt.show(fig)

dimension_dropdown.observe(render_dashboard, names='value')
start_picker.observe(render_dashboard, names='value')
end_picker.observe(render_dashboard, names='value')
display(widgets.HBox([start_picker, end_picker, dimension_dropdown]))
display(dashboard_output)
render_dashboard()

Output()